In [1]:
# Define your Google Drive folder path
drive_folder_path = '/content/drive/MyDrive/Gen AI and Langchain/ModelFiles/13_ANN/'

In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# load saved encoders and scaler
with open(drive_folder_path + 'label_encoder_gender.pkl', 'rb') as file:
  gender_label_encoder = pickle.load(file)

with open(drive_folder_path + 'onehot_encoder_geo.pkl', 'rb') as file:
  geo_onehot_encoder = pickle.load(file)

with open(drive_folder_path + 'scaler.pkl', 'rb') as file:
  scaler = pickle.load(file)


In [5]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [6]:
# transform data using loaded geo encoder
encoded_geography = geo_onehot_encoder.transform([[input_data['Geography']]])
geo_features_df = pd.DataFrame(encoded_geography,columns=geo_onehot_encoder.get_feature_names_out(['Geography']))
geo_features_df


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [7]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [8]:
input_df = pd.concat([input_df.reset_index(drop=True),geo_features_df],axis=1)
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [9]:
print(gender_label_encoder.classes_)

['Female' 'Male']


In [10]:
# transform data using loaded gender encoder
input_df['Gender'] = gender_label_encoder.transform([input_df['Gender']])
input_df

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,France,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [13]:
scaler_input = scaler.transform(input_df.drop(columns=['Geography']))
scaler_input

array([[-0.47154541,  0.90911166,  0.09477172, -0.69844549, -0.29010416,
         0.80510537,  0.63367318,  0.95214374, -0.84805047,  0.98019606,
        -0.57581067, -0.56349184]])

In [15]:
churn_model = load_model(drive_folder_path + 'churn_model.h5')

In [17]:
churn_probability = churn_model.predict(scaler_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step


In [19]:
if(churn_probability > 0.5):
  print("Customer is likely to churn")
else:
  print("Customer is not likely to churn")

Customer is not likely to churn
